# Bonus: SQL agent

Give the agent a **SQL tool** over the Chinook sample database (artists, albums, tracks). The model writes queries; your function runs them and returns rows (or error text so it can retry).

This is powerful and risky on real DBs — the demo uses a local SQLite file with a single `run` helper.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

`SQLDatabase.from_uri` wraps SQLite. Path is relative to the notebook working directory (`notebooks/module-2`).

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

Catch exceptions and return them as strings so the agent can fix the SQL. The last line is a direct tool invoke (no agent) to smoke-test the DB.

In [ ]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

## Agent

The question needs a join/aggregation ("most popular artist beginning with S"). Inspect `tool_calls` to see the SQL the model chose.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[sql_query]
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
# messages[-3] is typically the AI turn that issued the (last) SQL tool call
print(response["messages"][-3].tool_calls[0]['args']['query'])